# Train biped_rl on Kaggle (clone from GitHub)
1. Push this repo to GitHub first (the branch below must contain `scripts/` and `biped_rl/`).
2. Notebook settings: **Internet on**. Accelerator: none (physics runs on CPU; ~30 GB RAM either way).
3. Only if the repo becomes private: Add-ons -> Secrets -> add `GITHUB_TOKEN` (a read-only personal access token) and tick it for this notebook.
4. To continue a run, attach any one of: the previous version's output (**Add Input -> Your Work -> Notebooks**), a Kaggle Dataset holding a downloaded output zip (e.g. `results.zip`), or a dataset with `ckpt_<N>_steps.zip` + `ckpt_vecnormalize_<N>_steps.pkl`. It finds `runs/<RUN>/` (unzipping if needed), copies the whole run, and resumes from the newest checkpoint at the last curriculum level. Nothing to edit. With no input it trains from scratch.
5. *Save Version -> Save & Run All*. It stops itself at `MAX_HOURS` (Kaggle kills at 12 h). Results land in Output: `RobustStep/runs/<RUN>/` plus `<RUN>_progress.mp4` and `<RUN>_progress_chart.png`.

In [ ]:
REPO_URL = "https://github.com/MilesDao/RobustStep.git"
BRANCH = "main"
RUN, START_LEVEL, KP_SCALE, STEPS = "stand_b", 3, 3, 20_000_000  # ~614 steps/s on 4 vCPU, so 20M is about 9 h
MAX_HOURS = 10.5   # Kaggle hard limit is 12 h; leaves time for the video (<=50 min) + zipping

In [ ]:
import glob, os, re, shutil, subprocess
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    pass  # public repo, no token needed
os.chdir("/kaggle/working")
if not os.path.isdir("RobustStep/.git"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "RobustStep"], check=True)
os.chdir("/kaggle/working/RobustStep")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout, "cpus:", os.cpu_count())

In [ ]:
!pip install -q "mujoco>=3.2" "gymnasium>=1.0" "stable-baselines3>=2.3" imageio-ffmpeg
!pip install -q -e . --no-deps

In [ ]:
# continue from an attached input: a previous version's notebook output, a dataset of checkpoints,
# or a dataset holding a zip of a previous output (e.g. results.zip)
import zipfile
run = f"runs/{RUN}"
os.makedirs(run, exist_ok=True)
roots = ["/kaggle/input"]
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    try:
        with zipfile.ZipFile(z) as zf:
            if any(re.search(r"ckpt_\d+_steps\.zip$", n) for n in zf.namelist()):  # skips the checkpoints themselves
                dst = f"/tmp/unzipped/{len(roots)}"
                zf.extractall(dst); roots.append(dst); print("extracted", z)
    except zipfile.BadZipFile:
        pass
found = [f for r in roots for f in glob.glob(f"{r}/**/ckpt_*_steps.zip", recursive=True)]
mine = [f for f in found if os.path.basename(os.path.dirname(f)) == RUN]
ckpts = {int(re.findall(r"ckpt_(\d+)_steps", f)[0]): f for f in (mine or found)}
RESUME = ""
if ckpts:
    steps = max(ckpts)
    src = os.path.dirname(ckpts[steps])
    # copy the whole old run (log, monitor CSVs, older checkpoints) so the video spans every session;
    # copyfile, not copy2: files from /kaggle/input can carry read-only modes
    shutil.copytree(src, run, dirs_exist_ok=True, copy_function=shutil.copyfile)
    shutil.copyfile(ckpts[steps], f"{run}/latest.zip")
    shutil.copyfile(os.path.join(src, f"ckpt_vecnormalize_{steps}_steps.pkl"), f"{run}/vecnormalize.pkl")
    RESUME = f"--resume {run}/latest.zip"
    levels = re.findall(r"\[curriculum\] level (\d+)", open(f"{run}/train.log").read()) if os.path.exists(f"{run}/train.log") else []
    START_LEVEL = int(levels[-1]) if levels else START_LEVEL
    print(f"resuming from step {steps} at level {START_LEVEL} ({src})")
else:
    START_LEVEL = 0
    print("no checkpoint attached: training from scratch at level 0")

In [ ]:
N_ENVS = os.cpu_count()
# stop at MAX_HOURS with Ctrl-C so train.py still saves latest.zip
!PYTHONUNBUFFERED=1 timeout --signal=INT {MAX_HOURS}h python scripts/train.py --name {RUN} {RESUME} --start-level {START_LEVEL}     --kp-scale {KP_SCALE} --total-steps {STEPS} --n-envs {N_ENVS} --save-every 250000 2>&1 | tee -a runs/{RUN}/train.log | grep -E "level|speed |ep_len_mean|fps|total_timesteps|Error" 

In [ ]:
# progress video (one clip per checkpoint), rendered headless; capped at 50 min so the 12 h limit holds
VIDEO = True
if VIDEO:
    probe = "import mujoco; mujoco.Renderer(mujoco.MjModel.from_xml_string('<mujoco/>'), 64, 64)"
    for gl in ("egl", "osmesa"):
        if gl == "osmesa":
            subprocess.run("apt-get -qq install -y libosmesa6 > /dev/null", shell=True)
        env = dict(os.environ, MUJOCO_GL=gl, PYOPENGL_PLATFORM=gl)
        if subprocess.run(["python", "-c", probe], env=env, capture_output=True).returncode == 0:
            os.environ.update(MUJOCO_GL=gl, PYOPENGL_PLATFORM=gl); print("rendering with", gl); break
    else:
        raise RuntimeError("no headless OpenGL backend (egl/osmesa) works here")
    # a clip takes ~100 s to render and the cap cuts in at ~30 clips, so keep ~25 spread over the whole run
    EVERY = max(1, len(glob.glob(f"runs/{RUN}/ckpt_*_steps.zip")) // 25)
    VIDEO_OUT = f"/kaggle/working/{RUN}_progress.mp4"
    !timeout 50m python scripts/progress_video.py --runs runs/{RUN} --every {EVERY} --out {VIDEO_OUT} --title "G1 learns to walk ({RUN})"
